#### Dowoading Citbike Data Year 2025
1.We will try to download for one month.
2.Create a pipeline to download for the whole year.
3.Save the data in data/JC.

#### Loading Libraries

In [96]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

#### #Step 1: Define the Data Source


In [97]:
CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
PERIOD = "202510"

In [98]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"
url = f"{CITIBIKE_INDEX_URL}/{file_name}"
url

'https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip'

#### Step 2: Download Jersey Data

In [99]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)

zip_path = output_dir / file_name


## Dowloading the Zip file
url = f"{CITIBIKE_INDEX_URL}/{file_name}"

print(f"Downloading: {url}")

urlretrieve(url, zip_path)

print(f"Saved ZIP file to: {zip_path}")


with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)

print(f"Extracted files into: {output_dir}")

Downloading: https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip


Saved ZIP file to: ..\data\citibike\JC-202510-citibike-tripdata.zip
Extracted files into: ..\data\citibike


In [100]:
#Once we get it we can remove the zip file|
zip_path.unlink()
print("ZIP file removed.")

ZIP file removed.


#### Step 3: Read the Data


In [101]:
file_name  = 'JC-202510-citibike-tripdata.csv'
citibike_202510 = pd.read_csv(f'{output_dir}/{file_name}')
citibike_202510.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,929852F0DC04F49C,electric_bike,2025-10-08 06:59:53.767,2025-10-08 07:04:08.012,Warren St,JC006,Essex Light Rail,NaN,40.721124,-74.038051,NaN,NaN,member
1,7E6F8FADF45B1D8E,electric_bike,2025-10-08 08:13:15.096,2025-10-08 08:18:16.168,Newark Ave,JC032,Essex Light Rail,NaN,40.721525,-74.046305,NaN,NaN,member
2,9B679869FCA8F845,electric_bike,2025-10-09 10:10:26.020,2025-10-09 10:20:08.505,Liberty Light Rail,JC052,Newport PATH,NaN,40.711242,-74.055701,NaN,NaN,member
3,E2051DD457BC4076,electric_bike,2025-10-09 11:34:22.937,2025-10-09 11:48:49.501,Hoboken Ave at Monmouth St,JC105,Newport PATH,NaN,40.735208,-74.046964,NaN,NaN,member
4,239AC07371E414EC,electric_bike,2025-10-23 19:24:16.237,2025-10-23 19:28:22.592,Union St & Bergen Ave,JC122,Garfield Light Rail,JC119,40.715750,-74.078870,40.710526,-74.070112,member


#### Step 4: Downloading year 2025


In [102]:
#Here we need one function which will help as to create an iterator
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    year list of strings
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

In [103]:
periods  = period_iterator(year = ['2025'],start_m = 0, stop_m = 12)

periods

['202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [104]:
#Once we have the iterator, we can now go over the loop and try to:
#1.donload each period
#2.extract
#3.remove the zip file

In [105]:
for i in periods:
    file_patterns = [f"JC-{i}-citibike-tripdata.csv.zip", f"JC-{i}-citibike-tripdata.zip"]
    downloaded = False
    
    for file_name in file_patterns:
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"
        zip_path = output_dir / file_name
        
        try:
            # Adding verbose output to see which file it's trying to find
            print(f"Checking: {url}") 
            urlretrieve(url, zip_path)
            downloaded = True
            break 
        except (HTTPError, URLError):
            continue 

    if downloaded:
        # Extraction logic
        try:
            with ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(output_dir)
            zip_path.unlink()
            print(f"Success: {file_name}")
        except BadZipFile:
            print(f"Error: {file_name} is corrupted.")
    else:
        # This will print which month failed, helping you debug
        print(f"Warning: Could not find any valid file for period: {i}")

Checking: https://s3.amazonaws.com/tripdata/JC-202501-citibike-tripdata.csv.zip
Success: JC-202501-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202502-citibike-tripdata.csv.zip
Success: JC-202502-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202503-citibike-tripdata.csv.zip
Success: JC-202503-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202504-citibike-tripdata.csv.zip
Success: JC-202504-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202505-citibike-tripdata.csv.zip
Success: JC-202505-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202506-citibike-tripdata.csv.zip
Success: JC-202506-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202507-citibike-tripdata.csv.zip
Success: JC-202507-citibike-tripdata.csv.zip
Checking: https://s3.amazonaws.com/tripdata/JC-202508-citibike-tripdata.csv.zip
Success: JC-202508-citibike-tripdata.csv.zip


#### Step 5: Removing __MACOSX


In [106]:
import shutil

shutil.rmtree(output_dir / "__MACOSX")

#### Step 6: Concatinating


In [107]:
import glob

file_names = glob.glob(f'{output_dir}/*.csv')

dfs = []
cols = []
for file_name in file_names:
    df = pd.read_csv(file_name)
    print(df.columns, 2*"||",len(df.columns))

    cols.append(list(df.columns))
    dfs.append(df)

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      d

In [118]:
citibike_df = pd.concat(dfs)
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member


In [119]:
citibike_df.to_csv("../data/citibike/JC2025.csv", index = False)